# IMF PDF Batch Extractor

Extracts TOC entries, Structural Benchmarks, and SI chapter titles from IMF PDFs.

**Setup (one-time):** copy `.env.example` to `.env` and fill in `ANTHROPIC_API_KEY`,
`TOC_SCRIPT_DIR`, and `BASE_FOLDER`. Never put secrets or absolute paths directly in
notebook cells -- they end up committed to version control and break on other machines.

**Workflow:**
1. Run **Cell 1** -- loads config from `.env`
2. Set `COUNTRY` / `WORKERS` / `SI_ONLY` in the run cell if needed
3. Run all cells -- JSON results saved per country folder

In [1]:
import subprocess, sys as _sys
subprocess.run([_sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

import os
import sys
import glob
from pathlib import Path
from dotenv import load_dotenv, dotenv_values

# --- Robust .env discovery --------------------------------------------------
# Check content, not filename -- a file is only accepted if the required keys
# are actually filled in. override=True so re-running this cell after editing
# .env always picks up the latest values instead of stale ones already sitting
# in this kernel's environment.
_REQUIRED_DIR_KEYS = ("TOC_SCRIPT_DIR", "TOC_EXTRACTOR_DIR")
_NAME_PATTERNS = (".env", "env.example.txt", ".env.example.txt", "env.example", ".env.local")

def _is_filled_in(path: Path) -> bool:
    vals = dotenv_values(path)
    has_dir = any((vals.get(k) or "").strip() for k in _REQUIRED_DIR_KEYS)
    has_key = (vals.get("ANTHROPIC_API_KEY") or "").strip().startswith("sk-ant-")
    return has_dir and has_key

# Check the known project subfolder FIRST -- confirmed by hand as the
# correct one to use when more than one filled-in candidate exists.
_PREFERRED_DIR = Path.home() / "Downloads" / "pdf-section-extractor-main" / "Bilateral Surveillance Project"
_candidates = [Path(p) for name in _NAME_PATTERNS for p in [str(_PREFERRED_DIR / name)] if Path(p).is_file()]
_candidates += [Path(p) for name in _NAME_PATTERNS for p in [str(Path.cwd() / name)] if Path(p).is_file()]
if not _candidates:
    for name in _NAME_PATTERNS:
        _candidates += [Path(p) for p in glob.glob(str(Path.home() / "Downloads" / "**" / name), recursive=True)]

ENV_PATH = next((p for p in _candidates if _is_filled_in(p)), None)
if ENV_PATH is None:
    detail = "\n".join(f"  {p}  (filled in: {_is_filled_in(p)})" for p in _candidates) or "  (none found)"
    raise RuntimeError(f"No env file with a real ANTHROPIC_API_KEY and TOC_SCRIPT_DIR found. Checked:\n{detail}")

load_dotenv(ENV_PATH, override=True)
print(f"Loaded env from: {ENV_PATH}")

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
TOC_SCRIPT_DIR    = (os.environ.get("TOC_SCRIPT_DIR", "") or os.environ.get("TOC_EXTRACTOR_DIR", "")).strip()

if not ANTHROPIC_API_KEY.startswith("sk-ant-"):
    raise ValueError(f"ANTHROPIC_API_KEY missing or invalid in {ENV_PATH}.")
if not os.path.isdir(TOC_SCRIPT_DIR):
    raise ValueError(f"TOC_SCRIPT_DIR is not a valid directory: {TOC_SCRIPT_DIR!r}")

# --- Robust, VERIFIABLE engine discovery ------------------------------------
# Previously: `import imf_pdf_extraction_engine as extractor` -- an exact,
# hardcoded module name with no fallback. If the actively-maintained copy of
# this file is saved under a different name (e.g. toc_extractor.py) or in a
# subfolder, this would silently import a different, possibly stale file
# instead -- with no error, since Python happily imports whatever matches
# that exact name first on sys.path. Search recursively under both known
# names instead, and print exactly which file (and which fixes) got loaded so
# there's no ambiguity about what a batch run actually used.
_ENGINE_NAMES = ("imf_pdf_extraction_engine.py", "toc_extractor.py")
_engine_candidates = [
    p for name in _ENGINE_NAMES
    for p in glob.glob(os.path.join(os.path.abspath(TOC_SCRIPT_DIR), "**", name), recursive=True)
]
if not _engine_candidates:
    raise FileNotFoundError(
        f"No imf_pdf_extraction_engine.py or toc_extractor.py found anywhere under "
        f"{TOC_SCRIPT_DIR}. Top-level contents: {os.listdir(TOC_SCRIPT_DIR)}"
    )
if len(_engine_candidates) > 1:
    print(f"NOTE: found {len(_engine_candidates)} candidate engine file(s) -- using the most recent:")
    for p in sorted(_engine_candidates, key=os.path.getmtime, reverse=True):
        print(f"    {p}  (modified {os.path.getmtime(p):.0f})")

ENGINE_FILE = max(_engine_candidates, key=os.path.getmtime)

import importlib.util
for mod_name in list(sys.modules):
    if mod_name == "imf_pdf_extraction_engine":
        del sys.modules[mod_name]
spec = importlib.util.spec_from_file_location("imf_pdf_extraction_engine", ENGINE_FILE)
extractor = importlib.util.module_from_spec(spec)
sys.modules["imf_pdf_extraction_engine"] = extractor
spec.loader.exec_module(extractor)

extractor.set_api_key(ANTHROPIC_API_KEY)

print(f"\u2713 Engine loaded from: {ENGINE_FILE}")
print(f"\u2713 Version: {getattr(extractor, '__version__', 'MISSING')}")
print(f"\u2713 Has Structural Conditionality fix: {'conditionality' in extractor._SB_TABLE_TITLE_RE.pattern.lower()}")
print(f"\u2713 Has MDG narrative-heading fix: {hasattr(extractor, '_is_sb_table_page')}")
print(f"\u2713 Has Claude API retry logic: {'max_retries' in extractor._call_claude_vision.__code__.co_varnames}")
print(f"\u2713 Has Prior-Actions-boundary fix: {'not always present' in extractor._VISION_PROMPT_SB_FULL.lower()}")
print(f"\u2713 Has concurrency limiter: {hasattr(extractor, '_claude_semaphore')}")
print(f"\u2713 API key loaded: {ANTHROPIC_API_KEY[:12]}...")

Loaded env from: /Users/narminebenaissa/Downloads/pdf-section-extractor-main/Bilateral Surveillance Project/.env
NOTE: found 2 candidate engine file(s) -- using the most recent:
    /Users/narminebenaissa/Downloads/pdf-section-extractor-main/Bilateral Surveillance Project/imf_pdf_extraction_engine.py  (modified 1787923999)
    /Users/narminebenaissa/Downloads/pdf-section-extractor-main/Archive/toc_extractor.py  (modified 1783807393)
✓ Engine loaded from: /Users/narminebenaissa/Downloads/pdf-section-extractor-main/Bilateral Surveillance Project/imf_pdf_extraction_engine.py
✓ Version: 2.4.0-claude-api
✓ Has Structural Conditionality fix: True
✓ Has MDG narrative-heading fix: True
✓ Has Claude API retry logic: True
✓ Has Prior-Actions-boundary fix: True
✓ Has concurrency limiter: True
✓ API key loaded: sk-ant-api03...



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
# Previously: `!cp env.example.txt .env` -- this UNCONDITIONALLY overwrote
# your real, filled-in .env with whatever env.example.txt contains, every
# single time this notebook ran. If env.example.txt is a stale or unfilled
# template, this silently destroyed manual fixes made directly to .env
# between runs (Cell 1 above already loaded values into memory by this point,
# so it doesn't break the CURRENT run -- but it corrupts the file for the
# NEXT time you open this notebook).
#
# Cell 1 now does its own robust, content-aware .env discovery, so this
# cell no longer needs to do anything. Left here as a no-op so the cell
# numbering/flow doesn't shift.
print("Skipped -- .env is loaded directly by Cell 1, nothing to copy here.")

Skipped -- .env is loaded directly by Cell 1, nothing to copy here.


In [3]:
import sys, json, traceback, re, os
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# extraction engine already imported and key set in cell 1
extract_all_targets = extractor.extract_all_targets
print(f"✓ extraction engine ready  (version: {extractor.__version__})")


✓ extraction engine ready  (version: 2.4.0-claude-api)


In [4]:
_SI_NAME_RE = re.compile(r"_SI[_\d]*\.", re.IGNORECASE)

def _is_si_file(path: Path) -> bool:
    return bool(_SI_NAME_RE.search(path.name))

def _process_one(pdf_path_str: str) -> tuple:
    try:
        data = extract_all_targets(pdf_path_str)
        return pdf_path_str, "ok", data, None
    except Exception:
        return pdf_path_str, "error", None, traceback.format_exc()


def run_country(country_folder: Path, workers: int, si_only: bool):
    """Extract all PDFs in one country folder and save a toc_results JSON."""
    all_pdfs  = sorted(country_folder.rglob("*.pdf"))
    pdf_files = [p for p in all_pdfs if _is_si_file(p)] if si_only else all_pdfs

    label = f"SI PDF(s)" if si_only else "PDF(s)"
    if not pdf_files:
        print(f"  No {label} found in {country_folder.name}")
        if si_only and all_pdfs:
            print(f"  ({len(all_pdfs)} total PDFs present — none matched SI pattern)")
        return

    print(f"  {len(pdf_files)} {label} found — {workers} threads")

    all_results, failed = {}, []

    with tqdm(total=len(pdf_files), unit="pdf", dynamic_ncols=True) as pbar:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_process_one, str(p)): p for p in pdf_files}
            for fut in as_completed(futures):
                pdf_path = futures[fut]
                rel = str(pdf_path.relative_to(country_folder))
                pdf_str, status, data, tb = fut.result()

                if status == "ok":
                    date    = (data.get("document_date") or {}).get("raw", "?")
                    toc_n   = len(data.get("toc", []))
                    bench_n = len(data.get("benchmarks", []))
                    si_t    = data.get("si_title") or ""
                    si_disp = (si_t[:70] + "…") if len(si_t) > 70 else si_t
                    pbar.write(f"  ✓ {rel}  date={date}  toc={toc_n}  bench={bench_n}"
                               + (f"  SI={si_disp}" if si_t else ""))
                    all_results[pdf_str] = {
                        "file":      pdf_path.name,
                        "subfolder": str(pdf_path.parent.relative_to(country_folder)),
                        "path":      pdf_str,
                        "status":    "ok",
                        **data,
                        # pre-split si_chapters for json_to_excel
                        "si_chapters": [t.strip() for t in (data.get("si_title") or "").split(" | ")
                                        if t.strip()],
                    }
                else:
                    failed.append(rel)
                    short_err = tb.strip().splitlines()[-1] if tb else "unknown"
                    pbar.write(f"  ✗ {rel}  ERROR: {short_err}")
                    all_results[pdf_str] = {
                        "file": pdf_path.name, "subfolder": str(pdf_path.parent.relative_to(country_folder)),
                        "path": pdf_str, "status": "error",
                        "error": short_err, "traceback": tb,
                    }
                pbar.update(1)

    results_list = [all_results[str(p)] for p in pdf_files if str(p) in all_results]
    timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix       = "_SI_only" if si_only else ""
    out_path     = country_folder / f"toc_results{suffix}_{timestamp}.json"

    summary = {
        "run_timestamp": datetime.now().isoformat(),
        "country":       country_folder.name,
        "source_folder": str(country_folder),
        "si_only":       si_only,
        "workers_used":  workers,
        "total_files":   len(pdf_files),
        "successful":    sum(1 for r in results_list if r["status"] == "ok"),
        "failed":        sum(1 for r in results_list if r["status"] == "error"),
        "results":       results_list,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Saved → {out_path.name}")
    print(f"    {summary['successful']} succeeded  {summary['failed']} failed")
    if failed:
        print(f"    Failed: {failed}")


In [5]:
# ════════════════════════════════════════════════════════════════════════════════
# ▶  EDIT THIS CELL THEN RUN IT -- this is the only cell you need to change
# ════════════════════════════════════════════════════════════════════════════════
# BASE_FOLDER comes from .env (machine-specific path -- see .env.example)
BASE_FOLDER = os.environ.get("BASE_FOLDER", "")
if not BASE_FOLDER:
    raise ValueError("Set BASE_FOLDER in your .env file.")

# None = all countries │ "BEN" = one country │ ["BEN","TZA"] = subset
COUNTRY = ["GHA"]  # None = all countries │ str = one country │ list = subset

WORKERS = 3      # parallel threads (2-4 recommended)
SI_ONLY = False  # True = SI files only │ False = every PDF in the folder
# ════════════════════════════════════════════════════════════════════════════════

base = Path(BASE_FOLDER).expanduser().resolve()

if COUNTRY:
    countries = [COUNTRY] if isinstance(COUNTRY, str) else COUNTRY
    folders = [base / c for c in countries if (base / c).is_dir()]
else:
    folders = sorted([d for d in base.iterdir()
                      if d.is_dir() and d.name != "_Aggregate"
                      and any(d.rglob("*.pdf"))])

if not folders:
    print(f"No folders found under {base}")
else:
    print(f"Processing {len(folders)} folder(s): {[f.name for f in folders]}")
    print(f"SI_ONLY={SI_ONLY}  WORKERS={WORKERS}\n")
    for folder in tqdm(folders, desc="Countries", unit="country"):
        tqdm.write(f"\n── {folder.name} {'─'*40}")
        run_country(folder, workers=WORKERS, si_only=SI_ONLY)
    print("\nDone.")


Processing 1 folder(s): ['GHA']
SI_ONLY=False  WORKERS=3



Countries:   0%|          | 0/1 [00:00<?, ?country/s]


── GHA ────────────────────────────────────────
  8 PDF(s) found — 3 threads


Multiple captioned SB table clusters found: [[113, 115, 116], [119, 120, 121, 122]] -- using the last one ([119, 120, 121, 122]). If this is wrong, the document may need a closer look.
                                                     
Countries:   0%|          | 0/1 [00:27<?, ?country/s]

  ✓ GHA_2023_ECF_Request_Standalone_Pre-Combined.pdf  date=May 2023  toc=5  bench=1


                                                     
Countries:   0%|          | 0/1 [00:29<?, ?country/s]

... (9 outputs left out)

                                                     
Countries:   0%|          | 0/1 [01:35<?, ?country/s]

  ✓ GHA_2025_ECF_Review5_Standalone_Post_Combined_4.pdf  date=December 2025  toc=7  bench=1


                                                     
Countries: 100%|██████████| 1/1 [01:46<00:00, 106.02s/country]

  ✓ GHA_2026_ECF_Review6_PCI_Request_Article_IV_Combined.pdf  date=August 2026  toc=8  bench=1

  ✓ Saved → toc_results_20261029_144834.json
    8 succeeded  0 failed

Done.
